# ASR Server — Colab 测试（transcribe / start / stop）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/babel-world/asr-server/blob/main/notebooks/colab_transcribe_start_stop.ipynb)

在 Google Colab（T4 GPU）上验证以下三个接口：

- `POST /api/transcribe/start`
- `POST /api/transcribe`
- `POST /api/transcribe/stop`

## 使用步骤

1. 点击上方 **Open in Colab** 打开本 notebook。
2. 将测试 `.wav` 文件上传到 Colab 根目录（`/content/`），例如 `/content/sample.wav`。
3. 在下方配置单元中按需修改 `WAV_PATH`。
4. 依次运行全部单元：clone 仓库 → 安装依赖 → 启动服务 → 调用并断言三个接口。

## 环境变量

本 notebook 会写入项目根目录的 `.env`（参考 `.env.example`）：

| 变量 | 默认值 | 说明 |
|------|--------|------|
| `WHISPER_MODEL` | `base` | faster-whisper 模型尺寸（Colab T4 可试 `large-v3`） |
| `WHISPER_COMPUTE_TYPE` | `float16` | CTranslate2 GPU 计算类型 |

`WHISPER_DEVICE` 默认在代码中为 `cuda`，无需写入 `.env`。

模型路径解析：优先使用项目 `.models/`，其次 Hugging Face 默认缓存；若均不存在则下载到 `.models/`。

## 验证清单

| 步骤 | 接口 | 预期 |
|------|------|------|
| 1 | `POST /api/transcribe/start` | `200`，`loaded: true` |
| 2 | `POST /api/transcribe`（multipart `file`，`.wav`） | `200`，`transcribedText` 非空 |
| 3 | `POST /api/transcribe/stop` | `200`，`loaded: false` |

In [ ]:
# 配置
REPO_URL = "https://github.com/babel-world/asr-server.git"
REPO_DIR = "/content/asr-server"
WAV_PATH = "/content/sample.wav"  # 请改为你上传的 wav 文件名
BASE_URL = "http://127.0.0.1:19031"

# Colab T4 可使用更大模型；首次运行会下载到 .models/
WHISPER_MODEL = "base"  # 例如 large-v3
WHISPER_COMPUTE_TYPE = "float16"

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

# 安装 uv 并同步依赖（含 httpx 用于接口测试）
!pip install -q uv httpx
!uv sync --extra dev

env_path = Path(".env")
env_path.write_text(
    f"WHISPER_MODEL={WHISPER_MODEL}\nWHISPER_COMPUTE_TYPE={WHISPER_COMPUTE_TYPE}\n",
    encoding="utf-8",
)
print(f"Wrote {env_path.resolve()}")
print(env_path.read_text(encoding="utf-8"))

In [ ]:
import os
import subprocess
import time
from pathlib import Path

import httpx

# 后台启动 FastAPI 服务（读取 .env 中的 WHISPER_*）
server_proc = subprocess.Popen(
    ["uv", "run", "asr-server"],
    cwd=REPO_DIR,
    env={**os.environ, **dict(line.split("=", 1) for line in Path(".env").read_text().splitlines() if "=" in line)},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for _ in range(60):
    try:
        r = httpx.get(f"{BASE_URL}/docs", timeout=2.0)
        if r.status_code == 200:
            print("Server is ready:", BASE_URL)
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Server failed to start within 60s")

In [ ]:
import json
import time
from pathlib import Path

import httpx

wav = Path(WAV_PATH)
if not wav.is_file():
    raise FileNotFoundError(
        f"未找到 {wav}。请先将 .wav 文件上传到 Colab 根目录 /content/，并修改 WAV_PATH。"
    )

client = httpx.Client(timeout=600.0)
results = {}

# 1) POST /api/transcribe/start
t0 = time.perf_counter()
resp = client.post(f"{BASE_URL}/api/transcribe/start")
results["start"] = {
    "status": resp.status_code,
    "elapsed_ms": round((time.perf_counter() - t0) * 1000, 2),
    "body": resp.json(),
}
assert resp.status_code == 200, results["start"]
assert results["start"]["body"].get("loaded") is True

# 2) POST /api/transcribe
t0 = time.perf_counter()
with wav.open("rb") as f:
    resp = client.post(
        f"{BASE_URL}/api/transcribe",
        files={"file": (wav.name, f, "audio/wav")},
    )
results["transcribe"] = {
    "status": resp.status_code,
    "elapsed_ms": round((time.perf_counter() - t0) * 1000, 2),
    "body": resp.json(),
}
assert resp.status_code == 200, results["transcribe"]
assert results["transcribe"]["body"].get("transcribedText", "") != ""

# 3) POST /api/transcribe/stop
t0 = time.perf_counter()
resp = client.post(f"{BASE_URL}/api/transcribe/stop")
results["stop"] = {
    "status": resp.status_code,
    "elapsed_ms": round((time.perf_counter() - t0) * 1000, 2),
    "body": resp.json(),
}
assert resp.status_code == 200, results["stop"]
assert results["stop"]["body"].get("loaded") is False

print(json.dumps(results, ensure_ascii=False, indent=2))
print("\nAll checks passed.")

In [ ]:
# 可选：停止后台服务
try:
    server_proc.terminate()
    server_proc.wait(timeout=10)
    print("Server stopped.")
except NameError:
    print("server_proc not defined; skip.")